# Analyze Recorded DJ Set to Produce Chromagram Vectors

This notebook produces chromagram vectors, per time step, for the given recorded show.

Each chromagram vector covers the entire piano keyboard, per note. Stated more technically, each chromagram vector covers each note within the scientific pitch notation range C0 through B8. 

## Import useful libraries

In [1]:
import librosa
import pandas as pd

In [2]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import pyspark.sql.functions as F

In [3]:
from chromagram_functions import process_octave

## User settings

In [4]:
output_directory = 'output'
sampling_rate = 22050
hop_length = 512 * 300
spark_memory = '70G'

octave_min_inclusive = 0
octave_max_inclusive = 8

# location of the recorded DJ set which we want to analyze
filename_show = '/home/emily/Desktop/projects/dj/song_recognition/data/full-goth-set.mp3'

## Load recorded show and extract harmonic content

In [5]:
y_show, sr_show = librosa.load(filename_show)
y_show_harmonic, y_show_percussive = librosa.effects.hpss(y_show)

## Define function for extracting per time step features of the recorded DJ set

In [6]:
def process_show(y_harmonic, sampling_rate, hop_length, octave_min_inclusive = 0, octave_max_inclusive = 8):
    y_tick_labels = librosa.key_to_notes('C:major')  # we compute this every time for now
    results_list = []
    for octave_number in range(octave_min_inclusive, octave_max_inclusive + 1):
        chromagram = process_octave(y_harmonic, octave_number, sr = sampling_rate, hop_length = hop_length)
        df = pd.DataFrame(chromagram.T)
        df.columns = [x + str(octave_number) for x in y_tick_labels]
        results_list.append(df)
    df_all_octaves = pd.concat(results_list, axis = 1)
    return df_all_octaves

## Compute DJ set features per time step

In [7]:
pdf_show = process_show(
    y_show_harmonic,
    sampling_rate,
    hop_length,
    octave_min_inclusive = octave_min_inclusive,
    octave_max_inclusive = octave_max_inclusive,
)

In [8]:
pdf_show.head(3)

,C0,C♯0,D0,D♯0,E0,F0,F♯0,G0,G♯0,A0,...,D8,D♯8,E8,F8,F♯8,G8,G♯8,A8,A♯8,B8
0,0.299066,0.120263,0.070744,0.156555,0.114822,0.188053,0.149917,0.208183,0.329553,0.391382,...,0.262596,0.139924,0.207149,0.280173,0.504437,0.241148,0.054898,0.249797,0.213910,0.589303
1,0.300899,0.116360,0.067842,0.148446,0.112004,0.182992,0.152335,0.208832,0.333090,0.388091,...,0.267256,0.147961,0.219558,0.282087,0.500064,0.237202,0.064904,0.255851,0.208284,0.583291
2,0.302578,0.112255,0.064905,0.140594,0.109117,0.177985,0.154672,0.209521,0.336316,0.384682,...,0.271121,0.155805,0.233731,0.284787,0.494538,0.233651,0.075628,0.261388,0.203064,0.576492


## QA

In [9]:
len(pdf_show.index)

2168

In [10]:
len(pdf_show.dropna().index)

2168

## Identify the column names for the notes

In [11]:
pitch_columns = [x for x in pdf_show.columns if x != 'id']

## Enumerate the time steps

In [12]:
pdf_show['time_step'] = pdf_show.index

## Initialize a Spark session

In [13]:
conf = (
    SparkConf()
    .setAppName('AnalyzedTrackLibrary')
    .set('spark.executor.memory', spark_memory)
    .set('spark.driver.memory', spark_memory)
    .set('spark.driver.maxResultSize', spark_memory)
)

spark = SparkSession.builder.config(conf = conf).getOrCreate()

26/03/22 15:38:55 WARN Utils: Your hostname, emily-MS-7B96 resolves to a loopback address: 127.0.1.1; using 192.168.1.99 instead (on interface eno1)
26/03/22 15:38:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/22 15:38:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Convert Pandas DF to Spark DF and collapse pitch columns to a vector column

In [14]:
sdf_show = (
    spark
    .createDataFrame(pdf_show)
    .orderBy('time_step')
    .withColumn('array_show', F.array(*pitch_columns))
    .select('time_step', 'array_show')
)

In [15]:
sdf_show.show(5)

26/03/22 15:38:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+--------------------+
|time_step|          array_show|
+---------+--------------------+
|        0|[0.29906621575355...|
|        1|[0.30089887976646...|
|        2|[0.30257806181907...|
|        3|[0.30403009057044...|
|        4|[0.30521896481513...|
+---------+--------------------+
only showing top 5 rows



## Save for later

In [16]:
path_show_output = output_directory + '/show_vectors_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_show.write.mode('overwrite').parquet(path_show_output)

## Close Spark session

In [17]:
spark.stop()